# Train Super Bot locally (CPU, NVIDIA GPU, or Apple GPU)

This notebook runs locally on Windows, macOS, or Linux using PyTorch and PEFT LoRA. It detects CUDA, Apple MPS, or CPU automatically; a GPU is recommended, but CPU training is supported if you have enough system RAM and patience. Fine-tuning is not training a model from scratch. Gemma 2 2B is around 5 GB in half precision and considerably larger in float32; practical RAM/VRAM needs are higher, so small-memory computers may need a smaller compatible base model. Set `SUPER_BOT_BASE_MODEL` to change the default. Training examples stay on this computer; model/dataset downloads and the optional model upload require internet.

The first training stage downloads small streamed samples from the conversation and instruction datasets linked in the PDF: OpenAssistant, WildChat, UltraChat, SmolTalk, No Robots, Dolly 15K, CodeAlpaca, and Evol-Instruct-Code. Check each dataset's current terms and license before training; WildChat can contain personal content and automated redaction is imperfect. The Stack and Stack Exchange links are not bulk-downloaded as chat-training data: use small, license-filtered subsets as knowledge/reference material instead. Gemma access may require accepting Google's model terms on Hugging Face.

Later, the model can also learn from chats: a client must opt in with `/learn on`, and the server operator must set `SUPER_BOT_COLLECT_CHAT_TRAINING=1`. If you have an exported `user_chats.jsonl` (one `{"messages": [...]}` per line), place it at the `USER_CHAT_FILE` path printed below, and review it before training; if it is absent, only the public datasets are used. This notebook is self-contained and needs only pip packages, not any project files. Those exchanges do not modify live model weights after every message. Do not opt in sensitive or private conversations. Run the server only where chat participants understand this local training collection.

In [ ]:
%pip -q install "torch>=2.2" "transformers>=4.46,<5" "datasets>=3.2,<4" "peft>=0.13,<1" "accelerate>=1.2,<2" "huggingface_hub>=0.27,<2"

In [ ]:
import json
import os
import platform
import random
import sys
from pathlib import Path

import torch

MODEL_ID = os.getenv("SUPER_BOT_BASE_MODEL", "google/gemma-2-2b-it")
if os.getenv("SUPER_BOT_DATA_DIR", "").strip():
    SUPER_BOT_DIR = Path(os.environ["SUPER_BOT_DATA_DIR"]).expanduser()
elif platform.system() == "Darwin":
    SUPER_BOT_DIR = Path.home() / "Library" / "Application Support" / "super_bot"
elif os.name == "nt":
    SUPER_BOT_DIR = Path(os.getenv("APPDATA", Path.home() / "AppData" / "Roaming")) / "super_bot"
else:
    SUPER_BOT_DIR = Path(os.getenv("XDG_DATA_HOME", Path.home() / ".local" / "share")) / "super_bot"
DATA_DIR = SUPER_BOT_DIR / "training-data"
USER_CHAT_FILE = DATA_DIR / "user_chats.jsonl"
ADAPTER_DIR = SUPER_BOT_DIR / "adapters" / "superbot-transformers"
HF_REPO_ID = os.getenv("SUPER_BOT_HF_REPO_ID", "").strip()
MAX_PER_DATASET = 100
SELECTED_DATASETS = ["openassistant", "wildchat", "ultrachat", "smoltalk", "no_robots", "dolly", "codealpaca", "evol_code"]
ITERS = 200
MAX_LENGTH = 512
BATCH_SIZE = 1
SEED = 42
random.seed(SEED)
if torch.cuda.is_available():
    DEVICE = "cuda"
    COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    COMPUTE_DTYPE = torch.float16
else:
    DEVICE = "cpu"
    COMPUTE_DTYPE = torch.float32
print("Model:", MODEL_ID)
print("Training device:", DEVICE, "dtype:", COMPUTE_DTYPE)
if DEVICE == "cpu":
    print("CPU training works but can be very slow; allow substantial free RAM for the base model and activations.")
print("User chat training export:", USER_CHAT_FILE)
print("Per-dataset sample cap:", MAX_PER_DATASET)
print("Adapter output:", ADAPTER_DIR)
print("Hugging Face model repo:", HF_REPO_ID or "set SUPER_BOT_HF_REPO_ID to enable upload")

## Download and normalize the PDF's linked training datasets

Only compatible supervised conversation/instruction datasets are sampled for fine-tuning. Data is streamed and capped per dataset; the full datasets are not downloaded. Samples are normalized to user/assistant turns, deduplicated, and basic email/phone patterns are redacted. This does not guarantee that data is private or high-quality: review dataset licenses and examples before using them.

In [ ]:
import re
from datasets import load_dataset

# (repo, split, config) for each dataset linked in the PDF that suits chat fine-tuning.
PDF_SFT_DATASETS = {
    "openassistant": ("OpenAssistant/oasst1", "train", None),
    "wildchat": ("allenai/WildChat-1M", "train", None),
    "ultrachat": ("HuggingFaceH4/ultrachat_200k", "train_sft", None),
    "smoltalk": ("HuggingFaceTB/smoltalk", "train", "SFT"),
    "no_robots": ("HuggingFaceH4/no_robots", "train", None),
    "dolly": ("databricks/databricks-dolly-15k", "train", None),
    "codealpaca": ("HuggingFaceH4/CodeAlpaca_20K", "train", None),
    "evol_code": ("nickrosh/Evol-Instruct-Code-80k-v1", "train", None),
}
EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"(?<!\w)(?:\+?\d[\d().\-\s]{7,}\d)(?!\w)")
ROLES = {"human": "user", "prompter": "user", "user": "user", "gpt": "assistant", "model": "assistant", "assistant": "assistant"}

def clean(value):
    if not isinstance(value, str):
        return ""
    return PHONE_RE.sub("[PHONE REDACTED]", EMAIL_RE.sub("[EMAIL REDACTED]", value)).strip()

def normalize(turns):
    if not isinstance(turns, list):
        return None
    messages = []
    for turn in turns:
        if not isinstance(turn, dict):
            continue
        role = turn.get("role", turn.get("from", turn.get("speaker")))
        role = ROLES.get(role.casefold()) if isinstance(role, str) else None
        text = clean(turn.get("content", turn.get("text", turn.get("value"))))
        if role and text:
            messages.append({"role": role, "content": text})
    while messages and messages[0]["role"] != "user":
        messages.pop(0)
    if len(messages) < 2 or messages[-1]["role"] != "assistant":
        return None
    return messages

def instruction_pair(row):
    instruction = clean(row.get("instruction", row.get("prompt", "")))
    response = clean(row.get("response", row.get("output", row.get("completion", ""))))
    context = clean(row.get("context", row.get("input", "")))
    if not instruction or not response:
        return None
    if context:
        instruction = f"{instruction}\n\nContext:\n{context}"
    return [{"role": "user", "content": instruction}, {"role": "assistant", "content": response}]

def row_conversation(name, row):
    if name == "wildchat" and (row.get("toxic") is True or row.get("language") not in (None, "English")):
        return None
    return normalize(row.get("messages")) or normalize(row.get("conversation")) or instruction_pair(row)

def oasst_conversations(rows):
    by_id = {row["message_id"]: row for row in rows if isinstance(row.get("message_id"), str)}
    for row in by_id.values():
        if row.get("role") != "assistant" or row.get("deleted") is True or row.get("review_result") is False:
            continue
        if row.get("rank") not in (None, 0) or row.get("lang") not in (None, "en"):
            continue
        chain, current, seen_ids = [], row, set()
        while current is not None and len(chain) < 12 and current["message_id"] not in seen_ids:
            seen_ids.add(current["message_id"])
            chain.append({"role": current.get("role"), "content": current.get("text")})
            parent = current.get("parent_id")
            current = by_id.get(parent) if isinstance(parent, str) else None
        messages = normalize(chain[::-1])
        if messages:
            yield messages

def load_pdf_sft_examples(limit, names):
    examples, seen = [], set()
    for name in names:
        repo, split, config = PDF_SFT_DATASETS[name]
        kwargs = {"split": split, "streaming": True}
        if config:
            kwargs["name"] = config
        scan_limit = max(limit * 50, 5_000)
        rows = []
        for row in load_dataset(repo, **kwargs):
            rows.append(row)
            if len(rows) >= scan_limit:
                break
        if name == "openassistant":
            conversations = oasst_conversations(rows)
        else:
            conversations = (c for c in (row_conversation(name, r) for r in rows if isinstance(r, dict)) if c)
        count = 0
        for messages in conversations:
            key = tuple((m["role"], m["content"]) for m in messages)
            if key in seen:
                continue
            seen.add(key)
            examples.append({"messages": messages})
            count += 1
            if count >= limit:
                break
        print(f"{name}: {count} usable examples")
    return examples

def load_chat_examples(path):
    if not path.is_file():
        return []
    examples = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        row = json.loads(line)
        messages = normalize(row.get("messages") if isinstance(row, dict) else None)
        if not messages:
            raise ValueError(f"Invalid user chat at {path}:{line_number}")
        examples.append({"messages": messages})
    return examples

pdf_examples = load_pdf_sft_examples(MAX_PER_DATASET, SELECTED_DATASETS)
chat_examples = load_chat_examples(USER_CHAT_FILE)
examples = pdf_examples + chat_examples
random.shuffle(examples)
if len(examples) < 2:
    raise RuntimeError("Not enough usable training examples were loaded.")
print(f"PDF datasets: {len(pdf_examples):,}; opted-in user chats: {len(chat_examples):,}; total: {len(examples):,}")

## Prepare tokenized training data

Only conversation behavior and instruction-following belong in fine-tuning. Keep books, manuals, factual references, The Stack code archive, Stack Exchange exports, and frequently-changing facts in the separate SQLite knowledge library so they can be updated without retraining.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    if tokenizer.eos_token is None:
        raise ValueError("The selected tokenizer needs a pad or end-of-sequence token.")
    tokenizer.pad_token = tokenizer.eos_token
random.shuffle(examples)
validation_count = max(1, round(len(examples) * 0.1))
valid_examples = examples[:validation_count]
train_examples = examples[validation_count:]

def tokenize_example(example):
    token_ids = tokenizer.apply_chat_template(
        example["messages"], tokenize=True, add_generation_prompt=False,
        truncation=True, max_length=MAX_LENGTH,
    )
    if not token_ids:
        raise ValueError("The model chat template returned empty training tokens.")
    return {"input_ids": token_ids}

DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename, rows in (("train.jsonl", train_examples), ("valid.jsonl", valid_examples)):
    with (DATA_DIR / filename).open("w", encoding="utf-8") as output:
        for row in rows:
            output.write(json.dumps(row, ensure_ascii=False) + "\n")
train_dataset = Dataset.from_list(train_examples).map(tokenize_example, remove_columns=list(train_examples[0]))
valid_dataset = Dataset.from_list(valid_examples).map(tokenize_example, remove_columns=list(valid_examples[0]))
print(f"Train: {len(train_examples)}, validation: {len(valid_examples)}")

## Fine-tune with Transformers and LoRA

The default is intentionally short. Increase iterations only after checking memory, validation loss, and held-out examples. Review the exported user-chat JSONL before training; all examples in that file are otherwise included. This produces a standard PEFT LoRA adapter, not a standalone model. CPU, MPS, and CUDA are supported; CPU may take a long time, while CUDA/MPS memory limits depend on sequence length and model size.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=COMPUTE_DTYPE)
model.to(DEVICE)
model.config.use_cache = False
model.enable_input_require_grads()
model.gradient_checkpointing_enable()
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
training_args = TrainingArguments(
    output_dir=str(ADAPTER_DIR), max_steps=ITERS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=8, learning_rate=2e-4, warmup_steps=10,
    eval_strategy="steps", eval_steps=50, save_strategy="no",
    logging_steps=10, report_to="none", fp16=DEVICE == "cuda" and COMPUTE_DTYPE == torch.float16,
    bf16=DEVICE == "cuda" and COMPUTE_DTYPE == torch.bfloat16,
    use_mps_device=DEVICE == "mps", dataloader_pin_memory=DEVICE == "cuda",
)
trainer = Trainer(
    model=model, args=training_args, train_dataset=train_dataset, eval_dataset=valid_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)
trainer.train()
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved PEFT adapter to", ADAPTER_DIR)

## Use the trained adapter locally

The session server accepts an OpenAI-compatible phone/local-model endpoint. PEFT adapters can be loaded with `PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(MODEL_ID), ADAPTER_DIR)` and served using your preferred Transformers-compatible inference server. For llama.cpp/MLX serving, first merge or convert the adapter using that backend's supported conversion workflow; this adapter is not an MLX adapter or GGUF file. Keep local services bound to loopback. Set `PHONE_MODEL_BASE_URL=http://127.0.0.1:8080/v1` and `TEST_PHONE_API_KEY=local-superbot` in the repository's ignored `.env`, then start `python -m Ai.server`.

## Upload the trained adapter to Hugging Face

This uploads the **LoRA adapter**, not a merged standalone model. Set `SUPER_BOT_HF_REPO_ID` in the notebook environment to your Hugging Face repository (for example, `your-account/superbot-gemma2-adapter`). New repos are created **private**. If the repo already exists, the notebook asks you to confirm an update before adding a new commit; it refuses to upload to an existing public repo. Review the adapter and the base model's redistribution terms before sharing it or changing its visibility.

In [ ]:
from huggingface_hub import HfApi, get_token, login
from huggingface_hub.errors import RepositoryNotFoundError

if not HF_REPO_ID:
    raise ValueError("Set SUPER_BOT_HF_REPO_ID to your Hugging Face model repo ID before uploading.")
if not ADAPTER_DIR.is_dir():
    raise FileNotFoundError(f"Trained adapter directory does not exist: {ADAPTER_DIR}")
if not get_token():
    login()

adapter_readme = f'''---\nbase_model: {MODEL_ID}\nlibrary_name: peft\nlicense: other\ntags:\n- transformers\n- peft\n- lora\n- superbot\n---\n\n# Super Bot adapter\n\nThis repository contains a PEFT LoRA adapter trained with Transformers. It is not a standalone model; load it with the base model `{MODEL_ID}` and PEFT. The base model's access and license terms also apply. Review the training-data licenses and privacy requirements before redistributing this adapter.\n'''
(ADAPTER_DIR / "README.md").write_text(adapter_readme, encoding="utf-8")
api = HfApi()
try:
    repo_info = api.repo_info(repo_id=HF_REPO_ID, repo_type="model")
except RepositoryNotFoundError:
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=True)
    print("Created a new private repository:", HF_REPO_ID)
else:
    confirmation = input(
        f"Repository {HF_REPO_ID} already exists. Upload this adapter as a new commit? "
        "Type UPDATE to confirm: "
    )
    if confirmation != "UPDATE":
        raise RuntimeError("Upload cancelled; the existing repository was not changed.")
    if not repo_info.private:
        raise ValueError(f"Refusing to upload to public Hugging Face repo {HF_REPO_ID!r}; make it private first.")
upload_result = api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type="model",
    folder_path=str(ADAPTER_DIR),
    commit_message="Upload Super Bot LoRA adapter",
)
print("Uploaded private adapter:", upload_result)

In [ ]:
ADAPTER_DIR.is_dir(), sorted(path.name for path in ADAPTER_DIR.iterdir()) if ADAPTER_DIR.is_dir() else []